# Phase 1 Measurement Workflow (Debug/Bring-up)

This notebook is intentionally short and operational.
It uses real hardware only and focuses on a clear execution flow.


## Block 1: Setup

Set hardware/configuration parameters, processing options, save options,
and load the validated sweep configuration.


In [1]:
# BLOCK 1 - Setup (run once)
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "eis").exists() and (candidate / "USB6451").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from current working directory.")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from eis import (
    CaptureConditioningConfig,
    ExcitationConfig,
    HardwareConfig,
    ImpedanceProcessingConfig,
    RunSaveOptions,
    RunSelection,
    load_and_validate_config,
    plot_capture_fft_components,
    plot_capture_time_domain_components,
    plot_impedance_inverse_nyquist,
    plot_snr_vs_frequency,
    run_measure_process_save,
    run_preflight_only,
)

# ----------------------------- User Inputs -----------------------------
CONFIG_PATH = REPO_ROOT / "config_examples" / "config_phase1_example.xlsx"
BASE_OUTPUT_DIR = REPO_ROOT / "measurements"
SERIAL_NUMBER = f"PH1_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
USER_NAME = "operator"
DESCRIPTION = ""
REPEATS = 1
RUN_PREFLIGHT_DURING_SWEEP = False

# ----------------------------- Save Options ----------------------------
save_options = RunSaveOptions(
    write_metadata_bank_txt=True,
    write_metadata_bank_csv=True,
    write_metadata_report_html=True,
    write_metadata_report_pdf=False,
    write_description_file=True,
)
SAVE_PLOTS_PNG = True
SAVE_PLOTS_VECTOR = True

# ------------------------ Hardware/Excitation --------------------------
hardware = HardwareConfig(
    device="Dev1",
    ao_channel="ao0",
    ai_channels=("ai0", "ai7"),
    input_mode="diff",
)

excitation = ExcitationConfig(
    drive_mode="auto_from_current_rms",
    offset_v=0.0,
    manual_current_range="20A",
)

conditioning = CaptureConditioningConfig(
    settle_discard_s=0.15,
    extra_periods_for_trim=1,
    alignment_search_periods=1,
)

processing = ImpedanceProcessingConfig(
    method="fft",
    sine_fit_backend="numpy_lstsq",
    filter_mode="lowpass",
    lowpass_cutoff_hz=2000.0,
    shunt_resistance_ohm=0.008,
)

# ----------------------------- Preflight -------------------------------
PREFLIGHT_SAMPLE_RATE_SPS = None
PREFLIGHT_SAMPLES_PER_CHANNEL = None
PREFLIGHT_TEST_CURRENT_RMS_A = 10.0
PREFLIGHT_MANUAL_CURRENT_RANGE = "20A"
PREFLIGHT_SHUNT_RESISTANCE_OHM = 0.008
PREFLIGHT_SHUNT_TOLERANCE_PERCENT = 15.0
PREFLIGHT_CURRENT_CHANNEL_INDEX = 0
PREFLIGHT_SETTLE_DISCARD_S = 0.15

# ------------------------- Overlay Selection ---------------------------
LAST_N_FOR_OVERLAY = 3
SERIAL_FILTER_FOR_OVERLAY = SERIAL_NUMBER

sweep = load_and_validate_config(CONFIG_PATH)
DEFAULT_DEBUG_FREQUENCY_HZ = float(sweep.points[0].frequency_hz)

print(f"Repo root: {REPO_ROOT}")
print(f"Config rows: {len(sweep.points)}")
print("Run mode: REAL HARDWARE")
print(f"Serial number: {SERIAL_NUMBER}")
print(f"Default debug frequency: {DEFAULT_DEBUG_FREQUENCY_HZ:g} Hz")


Repo root: c:\Users\dinod\Desktop\EIS
Config rows: 14
Run mode: REAL HARDWARE
Serial number: PH1_20260303_232036
Default debug frequency: 12.54 Hz


## Block 2: Preflight Check Only

Run DAQ preflight validation without starting a measurement sweep.


In [2]:
# BLOCK 2 - Preflight Only (no measurement sweep)
preflight_only_result = run_preflight_only(
    sweep=sweep,
    hardware=hardware,
    excitation=excitation,
    sample_rate_sps=PREFLIGHT_SAMPLE_RATE_SPS,
    samples_per_channel=PREFLIGHT_SAMPLES_PER_CHANNEL,
    test_current_rms_a=PREFLIGHT_TEST_CURRENT_RMS_A,
    manual_current_range=PREFLIGHT_MANUAL_CURRENT_RANGE,
    shunt_resistance_ohm=PREFLIGHT_SHUNT_RESISTANCE_OHM,
    shunt_voltage_tolerance_percent=PREFLIGHT_SHUNT_TOLERANCE_PERCENT,
    current_channel_index=PREFLIGHT_CURRENT_CHANNEL_INDEX,
    settle_discard_s=PREFLIGHT_SETTLE_DISCARD_S,
)

print("Preflight-only completed")
print(f"  Sample rate: {preflight_only_result.sample_rate_sps:g} S/s")
print(f"  Samples/ch : {preflight_only_result.samples_per_channel}")
print(f"  Shape      : {preflight_only_result.measured_shape}")
print(f"  Message    : {preflight_only_result.message}")


DaqError: Device identifier is invalid.

Device Specified: Dev1

Task Name: _unnamedTask<0>
Status Code: -200220

## Block 3: Run + Process + Save

Run measurement sweep, compute impedance, save artifacts/metadata, and print summary.


In [ ]:
# BLOCK 3 - Run, process, save (+ validation summary)
run_bundle = run_measure_process_save(
    sweep=sweep,
    hardware=hardware,
    excitation=excitation,
    processing=processing,
    base_output_dir=BASE_OUTPUT_DIR,
    serial_number=SERIAL_NUMBER,
    user_name=USER_NAME,
    description=DESCRIPTION,
    repeats=REPEATS,
    run_preflight_during_sweep=RUN_PREFLIGHT_DURING_SWEEP,
    preflight_sample_rate_sps=PREFLIGHT_SAMPLE_RATE_SPS,
    preflight_samples_per_channel=PREFLIGHT_SAMPLES_PER_CHANNEL,
    preflight_test_current_rms_a=PREFLIGHT_TEST_CURRENT_RMS_A,
    preflight_manual_current_range=PREFLIGHT_MANUAL_CURRENT_RANGE,
    preflight_shunt_resistance_ohm=PREFLIGHT_SHUNT_RESISTANCE_OHM,
    preflight_shunt_voltage_tolerance_percent=PREFLIGHT_SHUNT_TOLERANCE_PERCENT,
    preflight_current_channel_index=PREFLIGHT_CURRENT_CHANNEL_INDEX,
    preflight_settle_discard_s=PREFLIGHT_SETTLE_DISCARD_S,
    conditioning=conditioning,
    save_options=save_options,
)

LAST_RUN_BUNDLE = run_bundle
LAST_RUN_ROOT = run_bundle.layout.root
LAST_RUN_PLOTS_DIR = run_bundle.layout.plots
LAST_RUN_SERIAL = SERIAL_NUMBER

available_frequency_repeat = sorted(
    {(float(c.frequency_hz), int(c.repeat_index)) for c in run_bundle.run_result.captures}
)

print("Run completed")
print(f"  Run folder      : {LAST_RUN_ROOT}")
print(f"  Captures        : {len(run_bundle.run_result.captures)}")
print(f"  Impedance rows  : {len(run_bundle.impedance_results)}")
print(f"  Raw artifacts   : {len(run_bundle.persisted_artifacts.capture_artifacts)}")
print(f"  Point summaries : {len(run_bundle.persisted_artifacts.point_summaries)}")
if run_bundle.run_result.preflight is None:
    print("  Sweep preflight : skipped")
else:
    print(f"  Sweep preflight : {run_bundle.run_result.preflight.message}")
print("  Saved files:")
for item in run_bundle.saved_paths:
    print(f"    - {item}")
print("  Available (frequency_hz, repeat_index):")
for frequency_hz, repeat_index in available_frequency_repeat:
    print(f"    - ({frequency_hz:.6g}, {repeat_index})")


## Block 4: Last Run Inverse Nyquist

Generate inverse Nyquist plot (`R` vs `-X`) for the most recent run.


In [ ]:
# BLOCK 4 - Inverse Nyquist for last run
if "LAST_RUN_BUNDLE" not in globals():
    raise RuntimeError("Run BLOCK 3 first so LAST_RUN_BUNDLE is available.")

selection_last = RunSelection(mode="last", serial_numbers=(LAST_RUN_SERIAL,))

inv_nyquist_png = (
    LAST_RUN_PLOTS_DIR / "last_run_inverse_nyquist.png"
    if SAVE_PLOTS_PNG
    else None
)
inv_nyquist_svg = (
    LAST_RUN_PLOTS_DIR / "last_run_inverse_nyquist.svg"
    if SAVE_PLOTS_VECTOR
    else None
)

plot_impedance_inverse_nyquist(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last,
    save_path=inv_nyquist_png,
)

if inv_nyquist_svg is not None:
    plot_impedance_inverse_nyquist(
        base_output_dir=BASE_OUTPUT_DIR,
        selection=selection_last,
        save_path=inv_nyquist_svg,
    )

print("Last-run inverse Nyquist generated")
if inv_nyquist_png is not None:
    print(f"  - {inv_nyquist_png}")
if inv_nyquist_svg is not None:
    print(f"  - {inv_nyquist_svg}")


## Block 5: Last Run SNR vs Frequency

Generate SNR vs frequency plot for the most recent run.


In [ ]:
# BLOCK 5 - SNR vs frequency for last run
if "LAST_RUN_BUNDLE" not in globals():
    raise RuntimeError("Run BLOCK 3 first so LAST_RUN_BUNDLE is available.")

selection_last = RunSelection(mode="last", serial_numbers=(LAST_RUN_SERIAL,))

snr_last_png = (
    LAST_RUN_PLOTS_DIR / "last_run_snr.png"
    if SAVE_PLOTS_PNG
    else None
)
snr_last_svg = (
    LAST_RUN_PLOTS_DIR / "last_run_snr.svg"
    if SAVE_PLOTS_VECTOR
    else None
)

_, _, _, checks_last = plot_snr_vs_frequency(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last,
    snr_source="current",
    threshold_db=20.0,
    good_region="above_threshold",
    save_path=snr_last_png,
)

if snr_last_svg is not None:
    plot_snr_vs_frequency(
        base_output_dir=BASE_OUTPUT_DIR,
        selection=selection_last,
        snr_source="current",
        threshold_db=20.0,
        good_region="above_threshold",
        save_path=snr_last_svg,
    )

print("Last-run SNR plot generated")
if snr_last_png is not None:
    print(f"  - {snr_last_png}")
if snr_last_svg is not None:
    print(f"  - {snr_last_svg}")
for item in checks_last:
    print(f"  Check: {item.run.root.name} passed={item.passed} min={item.min_snr_db:.2f} dB max={item.max_snr_db:.2f} dB")


## Block 6: Last N Runs Inverse Nyquist Overlay

Generate inverse Nyquist overlay for the newest `N` runs after serial filtering.


In [ ]:
# BLOCK 6 - Inverse Nyquist for last N runs
if "LAST_RUN_BUNDLE" not in globals():
    raise RuntimeError("Run BLOCK 3 first so output paths are available.")

N = int(LAST_N_FOR_OVERLAY)
if N < 1:
    raise ValueError("LAST_N_FOR_OVERLAY must be >= 1")

selection_last_n = RunSelection(
    mode="last_n",
    last_n=N,
    serial_contains=SERIAL_FILTER_FOR_OVERLAY,
)

inverse_last_n_png = LAST_RUN_PLOTS_DIR / f"last_{N}_inverse_nyquist.png"
inverse_last_n_svg = LAST_RUN_PLOTS_DIR / f"last_{N}_inverse_nyquist.svg"

_, _, runs_inv = plot_impedance_inverse_nyquist(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last_n,
    save_path=(inverse_last_n_png if SAVE_PLOTS_PNG else None),
)

if SAVE_PLOTS_VECTOR:
    plot_impedance_inverse_nyquist(
        base_output_dir=BASE_OUTPUT_DIR,
        selection=selection_last_n,
        save_path=inverse_last_n_svg,
    )

print(f"Last-{N} inverse Nyquist overlay generated")
print("  Runs used:")
for item in runs_inv:
    print(f"    - {item.root.name}")
if SAVE_PLOTS_PNG:
    print(f"  - {inverse_last_n_png}")
if SAVE_PLOTS_VECTOR:
    print(f"  - {inverse_last_n_svg}")


## Block 7: Last N Runs SNR vs Frequency Overlay

Generate SNR vs frequency overlay for the newest `N` runs after serial filtering.


In [ ]:
# BLOCK 7 - SNR vs frequency for last N runs
if "LAST_RUN_BUNDLE" not in globals():
    raise RuntimeError("Run BLOCK 3 first so output paths are available.")

N = int(LAST_N_FOR_OVERLAY)
if N < 1:
    raise ValueError("LAST_N_FOR_OVERLAY must be >= 1")

selection_last_n = RunSelection(
    mode="last_n",
    last_n=N,
    serial_contains=SERIAL_FILTER_FOR_OVERLAY,
)

snr_last_n_png = LAST_RUN_PLOTS_DIR / f"last_{N}_snr.png"
snr_last_n_svg = LAST_RUN_PLOTS_DIR / f"last_{N}_snr.svg"

_, _, runs_snr, checks_snr = plot_snr_vs_frequency(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last_n,
    snr_source="current",
    threshold_db=20.0,
    good_region="above_threshold",
    save_path=(snr_last_n_png if SAVE_PLOTS_PNG else None),
)

if SAVE_PLOTS_VECTOR:
    plot_snr_vs_frequency(
        base_output_dir=BASE_OUTPUT_DIR,
        selection=selection_last_n,
        snr_source="current",
        threshold_db=20.0,
        good_region="above_threshold",
        save_path=snr_last_n_svg,
    )

print(f"Last-{N} SNR overlay generated")
print("  Runs used:")
for item in runs_snr:
    print(f"    - {item.root.name}")
for item in checks_snr:
    print(f"  Check: {item.run.root.name} passed={item.passed} min={item.min_snr_db:.2f} dB max={item.max_snr_db:.2f} dB")
if SAVE_PLOTS_PNG:
    print(f"  - {snr_last_n_png}")
if SAVE_PLOTS_VECTOR:
    print(f"  - {snr_last_n_svg}")


## Block 8: Time-Domain Debug (Selected Frequency/Repeat)

Plot selected components in time domain for one chosen frequency and repeat.
Set selectors at the top of this cell.


In [ ]:
# BLOCK 8 - Time-domain debug plot for selected frequency/repeat/components
if "LAST_RUN_BUNDLE" not in globals():
    raise RuntimeError("Run BLOCK 3 first so LAST_RUN_BUNDLE is available.")

# Selectors (time-domain, independent from FFT block)
TD_FREQUENCY_HZ = DEFAULT_DEBUG_FREQUENCY_HZ
TD_REPEAT_INDEX = 1
TD_COMPONENTS = ("raw", "filtered", "fitted")
TD_CHANNEL_INDICES = (0, 1)


td_png = (
    LAST_RUN_PLOTS_DIR
    / f"debug_time_f{TD_FREQUENCY_HZ:.6g}_rep{TD_REPEAT_INDEX}.png"
    if SAVE_PLOTS_PNG
    else None
)
td_svg = (
    LAST_RUN_PLOTS_DIR / f"debug_time_f{TD_FREQUENCY_HZ:.6g}_rep{TD_REPEAT_INDEX}.svg"
    if SAVE_PLOTS_VECTOR
    else None
)

_, _, td_result = plot_capture_time_domain_components(
    run_result=LAST_RUN_BUNDLE.run_result,
    frequency_hz=TD_FREQUENCY_HZ,
    repeat_index=TD_REPEAT_INDEX,
    components=TD_COMPONENTS,
    processing_config=processing,
    channel_indices=TD_CHANNEL_INDICES,
    print_snr_table=True,
    save_path=td_png,
    save_vector_path=td_svg,
)

print("Time-domain debug plot generated")
print(
    f"  Selected: row={td_result.row_number}, repeat={td_result.repeat_index}, "
    f"f={td_result.frequency_hz:.6g} Hz"
)
if td_png is not None:
    print(f"  - {td_png}")
if td_svg is not None:
    print(f"  - {td_svg}")


## Block 9: Frequency-Domain Debug (Selected Frequency/Repeat)

Plot FFT magnitude of selected components for one chosen frequency and repeat.
Set selectors at the top of this cell.


In [ ]:
# BLOCK 9 - Frequency-domain debug plot for selected frequency/repeat/components
if "LAST_RUN_BUNDLE" not in globals():
    raise RuntimeError("Run BLOCK 3 first so LAST_RUN_BUNDLE is available.")

# Selectors (frequency-domain, independent from time-domain block)
FD_FREQUENCY_HZ = DEFAULT_DEBUG_FREQUENCY_HZ
FD_REPEAT_INDEX = 1
FD_COMPONENTS = ("raw", "filtered", "fitted")
FD_CHANNEL_INDICES = (0, 1)
FD_MAX_FREQUENCY_HZ = None

fd_png = (
    LAST_RUN_PLOTS_DIR
    / f"debug_fft_f{FD_FREQUENCY_HZ:.6g}_rep{FD_REPEAT_INDEX}.png"
    if SAVE_PLOTS_PNG
    else None
)
fd_svg = (
    LAST_RUN_PLOTS_DIR / f"debug_fft_f{FD_FREQUENCY_HZ:.6g}_rep{FD_REPEAT_INDEX}.svg"
    if SAVE_PLOTS_VECTOR
    else None
)

_, _, fd_result = plot_capture_fft_components(
    run_result=LAST_RUN_BUNDLE.run_result,
    frequency_hz=FD_FREQUENCY_HZ,
    repeat_index=FD_REPEAT_INDEX,
    components=FD_COMPONENTS,
    processing_config=processing,
    channel_indices=FD_CHANNEL_INDICES,
    max_frequency_hz=FD_MAX_FREQUENCY_HZ,
    print_snr_table=True,
    save_path=fd_png,
    save_vector_path=fd_svg,
)

print("Frequency-domain debug plot generated")
print(
    f"  Selected: row={fd_result.row_number}, repeat={fd_result.repeat_index}, "
    f"f={fd_result.frequency_hz:.6g} Hz"
)
if fd_png is not None:
    print(f"  - {fd_png}")
if fd_svg is not None:
    print(f"  - {fd_svg}")
